# Week 6 Spark Assignment

### Celebal Technologies Internship

**Name:** Eklavya

**Technology:** Apache Spark (PySpark)

**Assignment:** Week 6 - Spark Architecture and Data Processing


# Assignment Objective

The objective of this assignment is to understand Spark Architecture and perform efficient data processing using PySpark.

The assignment covers the following concepts:

- Understand Spark Architecture (Driver, Cluster Manager and Executors)
- Learn Lazy Evaluation and DAG (Lineage Graph)
- Read data from CSV and Parquet files
- Handle schemas properly
- Filter and select required data
- Modify DataFrames
- Apply Transformations and Actions
- Understand Wide Transformations
- Learn Predicate Pushdown
- Handle Null Values
- Build Spark Data Pipelines
- Save processed data into CSV and Parquet formats
- Learn Spark Performance Optimization techniques

# Step 1: Import Required Libraries

In [5]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

# Step 2: Create Spark Session

In [6]:
spark = SparkSession.builder \
    .appName("Week6 Spark Assignment") \
    .getOrCreate()

print("Spark Session Created Successfully")

Spark Session Created Successfully


# Step 3: Read the CSV Dataset

In [7]:
df = spark.read.csv(
    "Dataset/spark_assignment_dataset.csv",
    header=True,
    inferSchema=True
)

# Step 4: Display the Dataset

In [8]:
df.show(5)

+--------+----------+------------+---------------+------------+----------+-------+--------+--------+--------+-------+---------+--------+-------+------+----------+-----------+----------+----------+------------+
|order_id|product_id|    old_name|       category|sub_category|base_price|  price|quantity|  amount|discount| profit|   status|priority|user_id|region|      city|      state|order_date| ship_date|payment_mode|
+--------+----------+------------+---------------+------------+----------+-------+--------+--------+--------+-------+---------+--------+-------+------+----------+-----------+----------+----------+------------+
|       1|      1001|   Mobile v1|    Electronics|      Mobile|   1193.73|1649.03|       9|14841.27|     0.0| 4097.7|Completed|  Medium|10245.0|  East|   Kolkata|West Bengal|2026-03-09|2026-03-10| Net Banking|
|       2|      1002|      Pen v1|Office Supplies|         Pen|   1463.13|2108.64|       3| 6325.92|    0.15|1936.53|Completed|  Medium|11274.0|  West|      Pun

# Step 5: Display the Schema

In [9]:
df.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- old_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- sub_category: string (nullable = true)
 |-- base_price: double (nullable = true)
 |-- price: double (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- amount: double (nullable = true)
 |-- discount: double (nullable = true)
 |-- profit: double (nullable = true)
 |-- status: string (nullable = true)
 |-- priority: string (nullable = true)
 |-- user_id: double (nullable = true)
 |-- region: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- ship_date: date (nullable = true)
 |-- payment_mode: string (nullable = true)



# Step 6: Display Total Number of Records

In [14]:
print("Total Records :", df.count())

Total Records : 10000


# Step 7: Display Total Number of Columns

In [13]:
print("Total Columns :", len(df.columns))

Total Columns : 20


# Step 8: Display Column Names

In [12]:
print(df.columns)

['order_id', 'product_id', 'old_name', 'category', 'sub_category', 'base_price', 'price', 'quantity', 'amount', 'discount', 'profit', 'status', 'priority', 'user_id', 'region', 'city', 'state', 'order_date', 'ship_date', 'payment_mode']


# Step 9: Generate Summary Statistics

In [11]:
df.describe().show()

+-------+------------------+------------------+---------+---------------+------------+------------------+-----------------+-----------------+------------------+-------------------+------------------+---------+--------+------------------+------+---------+-----------+------------+
|summary|          order_id|        product_id| old_name|       category|sub_category|        base_price|            price|         quantity|            amount|           discount|            profit|   status|priority|           user_id|region|     city|      state|payment_mode|
+-------+------------------+------------------+---------+---------------+------------+------------------+-----------------+-----------------+------------------+-------------------+------------------+---------+--------+------------------+------+---------+-----------+------------+
|  count|             10000|             10000|     9814|          10000|       10000|             10000|            10000|            10000|             10000|

# Step 10: Check for Missing (Null) Values

In [10]:
df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
]).show()

+--------+----------+--------+--------+------------+----------+-----+--------+------+--------+------+------+--------+-------+------+----+-----+----------+---------+------------+
|order_id|product_id|old_name|category|sub_category|base_price|price|quantity|amount|discount|profit|status|priority|user_id|region|city|state|order_date|ship_date|payment_mode|
+--------+----------+--------+--------+------------+----------+-----+--------+------+--------+------+------+--------+-------+------+----+-----+----------+---------+------------+
|       0|         0|     186|       0|           0|         0|    0|       0|     0|       0|     0|     0|       0|    313|     0|   0|    0|         0|        0|           0|
+--------+----------+--------+--------+------------+----------+-----+--------+------+--------+------+------+--------+-------+------+----+-----+----------+---------+------------+



# Spark Architecture and Core Concepts


### Q1: Explain the roles of the Driver, Cluster Manager, and Executor in a Spark application.

Apache Spark follows a distributed computing architecture where different components work together to process large datasets efficiently.

**Driver**
- The Driver is the main program of a Spark application.
- It creates the Spark Session.
- It converts user code into jobs and stages.
- It builds the Directed Acyclic Graph (DAG).
- It schedules tasks and sends them to the Executors.
- It collects the results returned by the Executors.

**Cluster Manager**
- The Cluster Manager is responsible for managing the cluster resources.
- It allocates CPU and memory to the application.
- It launches Executors on worker nodes.
- It ensures efficient utilization of available resources.

**Executor**
- Executors are worker processes responsible for executing the tasks assigned by the Driver.
- They perform computations on the data.
- They can cache data in memory for faster execution.
- They send the execution results back to the Driver.

### Q2: How does Spark's Lazy Evaluation improve performance?

Spark uses Lazy Evaluation, which means transformations are not executed immediately. Instead, Spark records all transformations and waits until an Action is called.

When an Action is executed, Spark creates a Directed Acyclic Graph (DAG), optimizes the execution plan, removes unnecessary computations, and executes the job efficiently.

This optimization reduces disk I/O, minimizes data movement, and improves the performance of large-scale data processing.

In [15]:
# Transformation (No execution happens here)

lazy_df = df.filter(col("price") > 1000)

print("Transformation Created Successfully")

Transformation Created Successfully


In [16]:
# Action (Execution starts here)

lazy_df.show(5)

+--------+----------+------------+---------------+------------+----------+-------+--------+--------+--------+-------+---------+--------+-------+------+----------+-----------+----------+----------+------------+
|order_id|product_id|    old_name|       category|sub_category|base_price|  price|quantity|  amount|discount| profit|   status|priority|user_id|region|      city|      state|order_date| ship_date|payment_mode|
+--------+----------+------------+---------------+------------+----------+-------+--------+--------+--------+-------+---------+--------+-------+------+----------+-----------+----------+----------+------------+
|       1|      1001|   Mobile v1|    Electronics|      Mobile|   1193.73|1649.03|       9|14841.27|     0.0| 4097.7|Completed|  Medium|10245.0|  East|   Kolkata|West Bengal|2026-03-09|2026-03-10| Net Banking|
|       2|      1002|      Pen v1|Office Supplies|         Pen|   1463.13|2108.64|       3| 6325.92|    0.15|1936.53|Completed|  Medium|11274.0|  West|      Pun

### Understanding the Directed Acyclic Graph (DAG)

A Directed Acyclic Graph (DAG) is Spark's execution plan.

Every transformation performed on a DataFrame is recorded in the DAG instead of being executed immediately.

When an Action is called, Spark optimizes the DAG before execution, reducing unnecessary operations and improving performance.

The DAG also provides fault tolerance by allowing Spark to recompute only the lost partitions if an Executor fails.

### Q3: Read a CSV file located at "Dataset/spark_assignment_dataset.csv" with Header and Infer Schema enabled.

In [17]:
csv_df = spark.read.csv(
    "Dataset/spark_assignment_dataset.csv",
    header=True,
    inferSchema=True
)

csv_df.show(5)

+--------+----------+------------+---------------+------------+----------+-------+--------+--------+--------+-------+---------+--------+-------+------+----------+-----------+----------+----------+------------+
|order_id|product_id|    old_name|       category|sub_category|base_price|  price|quantity|  amount|discount| profit|   status|priority|user_id|region|      city|      state|order_date| ship_date|payment_mode|
+--------+----------+------------+---------------+------------+----------+-------+--------+--------+--------+-------+---------+--------+-------+------+----------+-----------+----------+----------+------------+
|       1|      1001|   Mobile v1|    Electronics|      Mobile|   1193.73|1649.03|       9|14841.27|     0.0| 4097.7|Completed|  Medium|10245.0|  East|   Kolkata|West Bengal|2026-03-09|2026-03-10| Net Banking|
|       2|      1002|      Pen v1|Office Supplies|         Pen|   1463.13|2108.64|       3| 6325.92|    0.15|1936.53|Completed|  Medium|11274.0|  West|      Pun

### Q4: Difference Between CSV and Parquet

| CSV | Parquet |
|------|----------|
| Row-based storage | Column-based storage |
| Larger file size | Compressed file size |
| Slower reading speed | Faster reading speed |
| Reads the complete row | Reads only required columns |
| Does not support Predicate Pushdown | Supports Predicate Pushdown |
| Higher storage requirement | Lower storage requirement |

Parquet is generally preferred in Spark applications because it stores data in a columnar format, supports compression, and improves query performance by reading only the required columns.

In [53]:
# Read CSV file
csv_df = spark.read.csv(
    "Dataset/spark_assignment_dataset.csv",
    header=True,
    inferSchema=True
)

# Read Parquet file
parquet_df = spark.read.parquet(
    "Output/Parquet_Output"
)

print("CSV Record Count:", csv_df.count())
print("Parquet Record Count:", parquet_df.count())

CSV Record Count: 10000
Parquet Record Count: 10000


### Observation

Both CSV and Parquet contain the same data, but Parquet is optimized for Spark because it stores data in a columnar format. It supports compression, faster queries, and Predicate Pushdown, making it more efficient than CSV for large-scale data processing.

# DataFrame Operations

### Q5: Select the columns `product_id` and `price` where the category is **Electronics**.

Spark provides the `select()` function to retrieve specific columns from a DataFrame and the `filter()` function to extract rows that satisfy a given condition.

In [18]:
electronics_df = df.select("product_id", "price") \
                   .filter(col("category") == "Electronics")

electronics_df.show()

+----------+-------+
|product_id|  price|
+----------+-------+
|      1001|1649.03|
|      1004|1953.41|
|      1013|3203.88|
|      1018| 675.22|
|      1026|4395.54|
|      1039|5064.15|
|      1047|  617.3|
|      1050|4221.11|
|      1056|6359.37|
|      1059| 5589.6|
|      1062|1873.72|
|      1066| 319.93|
|      1074| 708.51|
|      1078| 792.38|
|      1079| 776.77|
|      1081|2603.54|
|      1083|6602.72|
|      1089|4354.79|
|      1090|5336.55|
|      1096|4502.53|
+----------+-------+
only showing top 20 rows


### Q6: Rename the column `old_name` to `new_name` and cast the `price` column from String to Double.

The `withColumnRenamed()` function is used to rename an existing column, while the `cast()` function changes the data type of a column.

In this step, a new DataFrame named `transformed_df` is created. This DataFrame will be used throughout the remaining assignment after applying all required modifications.

In [19]:
transformed_df = (
    df.withColumnRenamed("old_name", "new_name")
      .withColumn("price", col("price").cast("double"))
      .withColumn("final_price", col("base_price") * 1.18)
)

In [20]:
transformed_df.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- new_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- sub_category: string (nullable = true)
 |-- base_price: double (nullable = true)
 |-- price: double (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- amount: double (nullable = true)
 |-- discount: double (nullable = true)
 |-- profit: double (nullable = true)
 |-- status: string (nullable = true)
 |-- priority: string (nullable = true)
 |-- user_id: double (nullable = true)
 |-- region: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- ship_date: date (nullable = true)
 |-- payment_mode: string (nullable = true)
 |-- final_price: double (nullable = true)



In [38]:
transformed_df.show(5)

+--------+----------+------------+---------------+------------+----------+-------+--------+--------+--------+-------+---------+--------+-------+------+----------+-----------+----------+----------+------------+------------------+
|order_id|product_id|    new_name|       category|sub_category|base_price|  price|quantity|  amount|discount| profit|   status|priority|user_id|region|      city|      state|order_date| ship_date|payment_mode|       final_price|
+--------+----------+------------+---------------+------------+----------+-------+--------+--------+--------+-------+---------+--------+-------+------+----------+-----------+----------+----------+------------+------------------+
|       1|      1001|   Mobile v1|    Electronics|      Mobile|   1193.73|1649.03|       9|14841.27|     0.0| 4097.7|Completed|  Medium|10245.0|  East|   Kolkata|West Bengal|2026-03-09|2026-03-10| Net Banking|         1408.6014|
|       2|      1002|      Pen v1|Office Supplies|         Pen|   1463.13|2108.64|  

### Q7: How does Spark use the Lineage Graph (DAG) to provide fault tolerance if a worker node fails?

Spark maintains a Lineage Graph, also known as a Directed Acyclic Graph (DAG), which records all transformations applied to a DataFrame.

If an Executor or worker node fails, Spark does not reload the entire dataset. Instead, it uses the Lineage Graph to recompute only the lost partitions from the original data and previous transformations.

This mechanism provides fault tolerance and ensures reliable execution of distributed applications.

### Q8: Filter the DataFrame where the `status` is **Completed** and the `amount` is greater than **1000**.

Multiple conditions can be combined using the logical **AND (`&`)** operator within the `filter()` function.

In [21]:
completed_orders = transformed_df.filter(
    (col("status") == "Completed") &
    (col("amount") > 1000)
)

completed_orders.show()

+--------+----------+------------+---------------+------------+----------+-------+--------+--------+--------+--------+---------+--------+-------+------+----------+-----------+----------+----------+------------+------------------+
|order_id|product_id|    new_name|       category|sub_category|base_price|  price|quantity|  amount|discount|  profit|   status|priority|user_id|region|      city|      state|order_date| ship_date|payment_mode|       final_price|
+--------+----------+------------+---------------+------------+----------+-------+--------+--------+--------+--------+---------+--------+-------+------+----------+-----------+----------+----------+------------+------------------+
|       1|      1001|   Mobile v1|    Electronics|      Mobile|   1193.73|1649.03|       9|14841.27|     0.0|  4097.7|Completed|  Medium|10245.0|  East|   Kolkata|West Bengal|2026-03-09|2026-03-10| Net Banking|         1408.6014|
|       2|      1002|      Pen v1|Office Supplies|         Pen|   1463.13|2108.6

### Q9: Explain the concept of Predicate Pushdown in Parquet.

Predicate Pushdown is an optimization technique supported by Parquet files.

Instead of loading the complete dataset into memory, Spark pushes filtering conditions directly to the Parquet storage layer. Only the required rows and columns are read from disk.

This reduces disk I/O, minimizes memory consumption, and significantly improves query performance when working with large datasets.

### Q10: Add a new column `final_price` which is the `base_price` multiplied by 1.18 (18% tax).

The `withColumn()` function creates a new column by applying an expression to existing columns.

The `final_price` column has already been created in **Q6** while transforming the DataFrame.

The following code displays the newly created column.

In [22]:
transformed_df.select(
    "product_id",
    "base_price",
    "final_price"
).show(10)

+----------+----------+------------------+
|product_id|base_price|       final_price|
+----------+----------+------------------+
|      1001|   1193.73|         1408.6014|
|      1002|   1463.13|         1726.4934|
|      1003|   2351.18|2774.3923999999997|
|      1004|   1462.07|1725.2425999999998|
|      1005|   1959.33|         2312.0094|
|      1006|   2546.23|         3004.5514|
|      1007|   2596.68|         3064.0824|
|      1008|   3631.54|         4285.2172|
|      1009|   1562.38|         1843.6084|
|      1010|   4405.74|         5198.7732|
+----------+----------+------------------+
only showing top 10 rows


# Transformations and Actions

### Q11: What is the difference between Transformations and Actions? Provide two examples of each.

Spark operations are divided into two categories:

**Transformations**
- Transformations create a new DataFrame from an existing one.
- They are lazily evaluated, meaning they are not executed until an Action is called.

**Examples:**
- `filter()`
- `select()`

**Actions**
- Actions trigger the execution of transformations and return a result or write data.

**Examples:**
- `show()`
- `count()`

In [23]:
# Transformation Examples

filtered_df = transformed_df.filter(col("price") > 1000)

selected_df = transformed_df.select(
    "product_id",
    "category",
    "price"
)

In [24]:
# Action Examples

print("Total Records:", transformed_df.count())

selected_df.show(5)

Total Records: 10000
+----------+---------------+-------+
|product_id|       category|  price|
+----------+---------------+-------+
|      1001|    Electronics|1649.03|
|      1002|Office Supplies|2108.64|
|      1003|       Clothing|2600.81|
|      1004|    Electronics|1953.41|
|      1005|Home Appliances|2929.76|
+----------+---------------+-------+
only showing top 5 rows


# Working with Parquet Files


### Q12: Load a Parquet file from `Output/Parquet_Output`, remove rows where `user_id` is null, and save the result as a CSV.

In this step, the Parquet file is loaded into a Spark DataFrame. Rows with null values in the `user_id` column are removed, and the cleaned data is saved in CSV format.

In [33]:
parquet_df = spark.read.parquet("Output/Parquet_Output")

parquet_df.show(5)

+--------+----------+------------+---------------+------------+----------+-------+--------+--------+--------+-------+---------+--------+-------+------+----------+-----------+----------+----------+------------+------------------+
|order_id|product_id|    new_name|       category|sub_category|base_price|  price|quantity|  amount|discount| profit|   status|priority|user_id|region|      city|      state|order_date| ship_date|payment_mode|       final_price|
+--------+----------+------------+---------------+------------+----------+-------+--------+--------+--------+-------+---------+--------+-------+------+----------+-----------+----------+----------+------------+------------------+
|       1|      1001|   Mobile v1|    Electronics|      Mobile|   1193.73|1649.03|       9|14841.27|     0.0| 4097.7|Completed|  Medium|10245.0|  East|   Kolkata|West Bengal|2026-03-09|2026-03-10| Net Banking|         1408.6014|
|       2|      1002|      Pen v1|Office Supplies|         Pen|   1463.13|2108.64|  

In [34]:
clean_df = parquet_df.filter(
    col("user_id").isNotNull()
)

clean_df.show(5)

+--------+----------+------------+---------------+------------+----------+-------+--------+--------+--------+-------+---------+--------+-------+------+----------+-----------+----------+----------+------------+------------------+
|order_id|product_id|    new_name|       category|sub_category|base_price|  price|quantity|  amount|discount| profit|   status|priority|user_id|region|      city|      state|order_date| ship_date|payment_mode|       final_price|
+--------+----------+------------+---------------+------------+----------+-------+--------+--------+--------+-------+---------+--------+-------+------+----------+-----------+----------+----------+------------+------------------+
|       1|      1001|   Mobile v1|    Electronics|      Mobile|   1193.73|1649.03|       9|14841.27|     0.0| 4097.7|Completed|  Medium|10245.0|  East|   Kolkata|West Bengal|2026-03-09|2026-03-10| Net Banking|         1408.6014|
|       2|      1002|      Pen v1|Office Supplies|         Pen|   1463.13|2108.64|  

In [35]:
clean_df.write.mode("overwrite").csv(
    "Output/CSV_Output",
    header=True
)

print("CSV file saved successfully.")

CSV file saved successfully.


### Q13: What is the difference between Client Mode and Cluster Mode?

**Client Mode**
- The Driver program runs on the user's local machine.
- Suitable for development and testing.
- If the client machine disconnects, the Spark application stops.

**Cluster Mode**
- The Driver program runs inside the cluster.
- Suitable for production environments.
- The application continues running even if the client disconnects.

Cluster Mode provides better fault tolerance and resource management for large-scale applications.

### Client Mode vs Cluster Mode

| Client Mode | Cluster Mode |
|--------------|--------------|
| Driver runs on local machine | Driver runs inside the cluster |
| Best for development | Best for production |
| Stops if client disconnects | Continues running independently |
| Easier to debug | Better scalability and fault tolerance |

# Performance Concepts and Spark Data Pipeline

### Q14: Filter the dataset for rows where the `region` is **North** OR the `priority` is **High**.

The `filter()` function can combine multiple conditions using the logical **OR (`|`)** operator to retrieve rows that satisfy at least one condition.

In [26]:
north_high_df = transformed_df.filter(
    (col("region") == "North") |
    (col("priority") == "High")
)

north_high_df.show()

+--------+----------+------------+---------------+------------+----------+-------+--------+--------+--------+--------+---------+--------+-------+------+-----------+-----------+----------+----------+------------+------------------+
|order_id|product_id|    new_name|       category|sub_category|base_price|  price|quantity|  amount|discount|  profit|   status|priority|user_id|region|       city|      state|order_date| ship_date|payment_mode|       final_price|
+--------+----------+------------+---------------+------------+----------+-------+--------+--------+--------+--------+---------+--------+-------+------+-----------+-----------+----------+----------+------------+------------------+
|       3|      1003|   Jacket v1|       Clothing|      Jacket|   2351.18|2600.81|       7|18205.67|     0.0| 1747.41|Completed|  Medium|12963.0| North|     Jaipur|  Rajasthan|2025-10-28|2025-11-04|  Debit Card|2774.3923999999997|
|       4|      1004|   Laptop v1|    Electronics|      Laptop|   1462.07|19

### Q15: Why is it safer to use `show(5)` instead of `collect()` on a multi-terabyte dataset?

The `show(5)` function displays only the first few rows of a DataFrame, making it safe for exploring very large datasets.

The `collect()` function transfers the entire dataset from all Executors to the Driver's memory. For very large datasets, this may consume excessive memory and can cause the Driver program to fail.

Therefore, `show()` is recommended for data exploration, while `collect()` should only be used when the dataset is small.

In [27]:
transformed_df.show(5)

print("Avoid using collect() on very large datasets because it loads all data into the Driver memory.")

+--------+----------+------------+---------------+------------+----------+-------+--------+--------+--------+-------+---------+--------+-------+------+----------+-----------+----------+----------+------------+------------------+
|order_id|product_id|    new_name|       category|sub_category|base_price|  price|quantity|  amount|discount| profit|   status|priority|user_id|region|      city|      state|order_date| ship_date|payment_mode|       final_price|
+--------+----------+------------+---------------+------------+----------+-------+--------+--------+--------+-------+---------+--------+-------+------+----------+-----------+----------+----------+------------+------------------+
|       1|      1001|   Mobile v1|    Electronics|      Mobile|   1193.73|1649.03|       9|14841.27|     0.0| 4097.7|Completed|  Medium|10245.0|  East|   Kolkata|West Bengal|2026-03-09|2026-03-10| Net Banking|         1408.6014|
|       2|      1002|      Pen v1|Office Supplies|         Pen|   1463.13|2108.64|  

## Handling Null Values

Handling missing values is an important step in data preprocessing. Spark provides functions such as `dropna()` and `fillna()` to remove or replace null values.

In [28]:
transformed_df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in transformed_df.columns
]).show()

+--------+----------+--------+--------+------------+----------+-----+--------+------+--------+------+------+--------+-------+------+----+-----+----------+---------+------------+-----------+
|order_id|product_id|new_name|category|sub_category|base_price|price|quantity|amount|discount|profit|status|priority|user_id|region|city|state|order_date|ship_date|payment_mode|final_price|
+--------+----------+--------+--------+------------+----------+-----+--------+------+--------+------+------+--------+-------+------+----+-----+----------+---------+------------+-----------+
|       0|         0|     186|       0|           0|         0|    0|       0|     0|       0|     0|     0|       0|    313|     0|   0|    0|         0|        0|           0|          0|
+--------+----------+--------+--------+------------+----------+-----+--------+------+--------+------+------+--------+-------+------+----+-----+----------+---------+------------+-----------+



In [29]:
clean_df = transformed_df.dropna()

print("Rows after removing null values:", clean_df.count())

Rows after removing null values: 9506


## Wide Transformations and Shuffle

A **Wide Transformation** is a transformation where data is shuffled across partitions before producing the output.

Examples:
- groupBy()
- join()
- distinct()

Shuffling increases network communication and execution time, so it should be minimized whenever possible.

In [ ]:
category_sales = transformed_df.groupBy("category").agg(
    sum("amount").alias("total_sales")
)

category_sales.show()


+---------------+--------------------+
|       category|         total_sales|
+---------------+--------------------+
|    Electronics| 3.461224637000001E7|
|       Clothing|3.5760830330000035E7|
|Office Supplies| 3.480548281999999E7|
|Home Appliances| 3.762839318999998E7|
|      Furniture|3.5499573080000006E7|
+---------------+--------------------+



In [40]:
category_sales.explain(mode="formatted")

== Physical Plan ==
AdaptiveSparkPlan (5)
+- HashAggregate (4)
   +- Exchange (3)
      +- HashAggregate (2)
         +- Scan csv  (1)


(1) Scan csv 
Output [2]: [category#20, amount#25]
Batched: false
Location: InMemoryFileIndex [file:/c:/Users/eklav/OneDrive/Desktop/Week6_Spark_Assignment/Dataset/spark_assignment_dataset.csv]
ReadSchema: struct<category:string,amount:double>

(2) HashAggregate
Input [2]: [category#20, amount#25]
Keys [1]: [category#20]
Functions [1]: [partial_sum(amount#25)]
Aggregate Attributes [1]: [sum#2309]
Results [2]: [category#20, sum#2310]

(3) Exchange
Input [2]: [category#20, sum#2310]
Arguments: hashpartitioning(category#20, 200), ENSURE_REQUIREMENTS, [plan_id=659]

(4) HashAggregate
Input [2]: [category#20, sum#2310]
Keys [1]: [category#20]
Functions [1]: [sum(amount#25)]
Aggregate Attributes [1]: [sum(amount#25)#2306]
Results [2]: [category#20, sum(amount#25)#2306 AS total_sales#2284]

(5) AdaptiveSparkPlan
Output [2]: [category#20, total_sales#2284]
Ar

## Complete Spark Data Pipeline

The following pipeline demonstrates a complete Spark workflow:

**Read → Transform → Filter → Handle Null Values → Write**

This follows the ETL (Extract, Transform, Load) approach commonly used in Data Engineering.

In [49]:
pipeline_df = (
    spark.read.csv(
        "Dataset/spark_assignment_dataset.csv",
        header=True,
        inferSchema=True
    )
    .withColumnRenamed("old_name", "new_name")
    .withColumn("price", col("price").cast("double"))
    .withColumn("final_price", col("base_price") * 1.18)
    .filter(col("status") == "Completed")
    .filter(col("user_id").isNotNull())
    .select(
        "order_id",
        "product_id",
        "new_name",
        "category",
        "price",
        "final_price",
        "status",
        "user_id"
    )
)

pipeline_df.show(10)

+--------+----------+------------+---------------+-------+------------------+---------+-------+
|order_id|product_id|    new_name|       category|  price|       final_price|   status|user_id|
+--------+----------+------------+---------------+-------+------------------+---------+-------+
|       1|      1001|   Mobile v1|    Electronics|1649.03|         1408.6014|Completed|10245.0|
|       2|      1002|      Pen v1|Office Supplies|2108.64|         1726.4934|Completed|11274.0|
|       3|      1003|   Jacket v1|       Clothing|2600.81|2774.3923999999997|Completed|12963.0|
|       4|      1004|   Laptop v1|    Electronics|1953.41|1725.2425999999998|Completed|11717.0|
|       5|      1005|Microwave v1|Home Appliances|2929.76|         2312.0094|Completed|10459.0|
|       6|      1006|      Fan v1|Home Appliances|3687.21|         3004.5514|Completed|12021.0|
|       7|      1007|  Printer v1|Office Supplies|2832.74|         3064.0824|Completed|11311.0|
|       9|      1009|Microwave v1|Home A

In [50]:
print("Total Records After Pipeline:", pipeline_df.count())

Total Records After Pipeline: 7266



## Save Processed Data

The processed DataFrame is saved in both CSV and Parquet formats for further analysis and storage.

In [51]:
pipeline_df.write.mode("overwrite").csv(
    "Output/Final_CSV",
    header=True
)

pipeline_df.write.mode("overwrite").parquet(
    "Output/Final_Parquet"
)

print("Processed data saved successfully in CSV and Parquet formats.")

Processed data saved successfully in CSV and Parquet formats.


In [52]:
print("CSV and Parquet files saved successfully.")

print("CSV Records:", spark.read.csv(
    "Output/Final_CSV",
    header=True
).count())

print("Parquet Records:", spark.read.parquet(
    "Output/Final_Parquet"
).count())

CSV and Parquet files saved successfully.
CSV Records: 7266
Parquet Records: 7266


# Performance Insights

1. Spark uses Lazy Evaluation to optimize execution by delaying computations until an action is called.
2. Parquet provides better performance than CSV because it stores data in a columnar format and supports compression.
3. Predicate Pushdown reduces disk I/O by reading only the required rows and columns.
4. Wide transformations involve shuffling data across partitions, which can impact performance.
5. The `show()` function is preferred over `collect()` for exploring large datasets because it avoids loading all data into the Driver's memory.
6. Handling null values improves data quality before performing analysis.
7. Building a pipeline simplifies data processing by combining multiple transformations into a structured workflow.
8. Spark distributes data across multiple Executors, enabling parallel processing and improving scalability for large datasets.

# Conclusion

In this assignment, we explored the architecture and execution model of Apache Spark and learned how to process large datasets efficiently using PySpark.

The assignment covered Spark Architecture, Lazy Evaluation, DAG, schema handling, DataFrame transformations, filtering, null value handling, CSV and Parquet formats, performance optimization techniques, and the development of an end-to-end Spark data pipeline.

These concepts demonstrate how Spark enables scalable and efficient data processing for real-world Data Engineering applications.